<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/Kalman_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and Setup

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy
from sympy import symbols
from sympy import *

import plotly.graph_objects as go
import plotly.express as px
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
from IPython.display import display, Math, Latex

In [ ]:
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

# The Kalman Filter on $\mathbb{R}^n$

Consider the linear Gaussian process:
\begin{align*}
x_k &= A_{k-1}\,x_{k-1} + G_{k-1}\,w_{k-1}, \\
y_k &= H_k\,x_k + z_k,
\end{align*}
where $w_k \sim \mathscr{N}(0,\Sigma_p)$ and $z_k \sim \mathscr{N}(0,\Sigma_m)$ are mutually independent white noise sequences, also independent of the initial state $x_0 \sim \mathscr{N}(m_0, P_0)$.
Here $y_k$ denotes the random variable representing the measurement at time step $k$, while $y_k^{\mathrm{obs}}$ denotes its observed numerical realization.

---

**Prediction Step (Time Update):**

Define the filter model
\begin{align*}
x^{-}_k &= A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \\
y^{-}_k &= H_k\,x^{-}_k + z_k,
\end{align*}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}),$$


Because the filter equation is linear and the process noise is Gaussian, the predicted state $x^{-}_k$ is also Gaussian. Taking the expectation of the state equation yields the predicted mean:


$$m_k^- \triangleq \mathbb{E}[x^{-}_k] = A_{k-1}\mathbb{E}[x^{+}_{k-1}] + G_{k-1}\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1},$$


since $\mathbb{E}[w_{k-1}] = 0$.

The predicted covariance $P_k^-$ is computed by applying the variance operator to the state equation:
\begin{align*}
P_k^- &\triangleq \text{Var}(x^{-}_k) \\
&= A_{k-1}\text{Var}(x^{+}_{k-1})A_{k-1}^T + G_{k-1}\text{Var}(w_{k-1})G_{k-1}^T \\
&= A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T.
\end{align*}
Thus, prior to incorporating the new measurement, our belief of the state is characterized by the prior distribution:


$$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$

---

**Measurement Prediction:**

Prior to its physical realization, the upcoming measurement at time step $k$ is treated as the random variable $y_k$. Based on our prior state belief, its expected value is:


$$\mathbb{E}[y^{-}_k] = H_k \mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^-,$$


and its variance is:


$$\text{Var}(y^{-}_k) = H_k \text{Var}(x_k^-) H_k^T + \text{Var}(z_k) = H_k P_k^- H_k^T + \Sigma_m.$$

---

**Joint Prior Distribution:**

The cross-covariance between the predicted state $x_k^-$ and the anticipated measurement random variable $y_k$ is evaluated as:


$$\text{Cov}(x_k^-, y^{-}_k) = \text{Cov}(x_k^-, H_k x_k^- + z_k) = P_k^- H_k^T.$$

Therefore, the joint distribution of the predicted state and the upcoming measurement is a block-structured multivariate Gaussian:
$$
\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}
\right).
$$

---

**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$


where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq P_k^- H_k^T (H_k P_k^- H_k^T + \Sigma_m)^{-1}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - H_k m_k^-), \\
P_k &= (I - K_k H_k) P_k^-.
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.

You may refer to this note on [multivariate Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) for the exact details of extracting the mean and the covariance of the conditional distribution.

---

**Error Dynamics**

For clarity, define the actual estimation error by
$$
\varepsilon_k \triangleq x_k - m_k,
\qquad
\varepsilon_k^- \triangleq x_k - m_k^- .
$$

From the measurement update,
$$
m_k = m_k^- + K_k\left(y_k^{\mathrm{obs}} - H_km_k^-\right),
$$
and using
$$
y_k^{\mathrm{obs}} = H_kx_k + z_k,
$$
we obtain
$$
\begin{aligned}
\varepsilon_k
&= x_k - m_k \\
&= x_k - m_k^- - K_k\left(H_kx_k + z_k - H_km_k^-\right) \\
&= \left(I-K_kH_k\right)(x_k-m_k^-) - K_kz_k \\
&= \left(I-K_kH_k\right)\varepsilon_k^- - K_kz_k .
\end{aligned}
$$

Moreover, from the prediction step,
$$
x_k = A_{k-1}x_{k-1} + G_{k-1}w_{k-1},
\qquad
m_k^- = A_{k-1}m_{k-1},
$$
we have
$$
\varepsilon_k^-
=
A_{k-1}\varepsilon_{k-1}
+
G_{k-1}w_{k-1}.
$$

Therefore,
$$
\boxed{
\varepsilon_k
=
\left(I-K_kH_k\right)A_{k-1}\varepsilon_{k-1}
+
\left(I-K_kH_k\right)G_{k-1}w_{k-1}
-
K_kz_k .
}
$$

# Examples

## 1-D Example

Consider the scalar linear-Gaussian filter model:
\begin{aligned}
x^-_k &= a\,x^+_{k-1} + w_{k-1},\qquad w_{k-1}\sim\mathscr N(0,\Sigma_q),\\
y^-_k &= h\,x^-_k + z_k,\qquad\;\;\;\; z_k\sim\mathscr N(0,\Sigma_r),
\end{aligned}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}).$$


**Prediction:**

Let $$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$ From the above filter model we have:
\begin{aligned}
m_k^- &= a\,m_{k-1},\\
P_k^- &= a^2 P_{k-1} + \Sigma_q.
\end{aligned}

**Predicted Measurment:**

From the filter model we have that
$$y_k^- \sim \mathscr{N}(hm_k^-, h^2P_k^-+\Sigma_r).$$


Thus
$$
\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ h m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- h \\ P_k^-h & h^2P_k^-+\Sigma_r \end{bmatrix}
\right).
$$



**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$

where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq \frac{P_k^- h }{(h^2P_k^- + \Sigma_r)}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - h m_k^-), \\
P_k &= (1 - K_k h)\,P_k^-
= \Bigl(1 - \frac{P_k^- h^2}{(h^2P_k^- + \Sigma_r)}\Bigr) P_k^- .
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.



In [ ]:
import numpy as np

def make_cv1d(dt: float = 0.1,
              q: float = 1e-2,
              r: float = 1e-1,
              x0=(0.0, 1.0),
              seed: int | None = None) -> sims.LinearGaussianSystemSyms:
    """
    Build a 1D constant-velocity linear Gaussian system:

        x_k = A x_{k-1} + w_{k-1},      w ~ N(0, Q)
        y_k = H x_k       + z_k,        z ~ N(0, R)

    State: x = [position, velocity]^T  (n=2)
    Measurement: y = position (scalar, p=1)

    Parameters
    ----------
    dt : float
        Sampling period Δt.
    q : float
        Continuous white-acceleration noise intensity (process noise scale).
        Discrete-time Q = q * [[dt^3/3, dt^2/2],
                               [dt^2/2, dt     ]].
    r : float
        Measurement noise std. R = [[r^2]] (scalar variance).
    x0 : tuple[float, float]
        Initial state (position, velocity).
    seed : int | None
        Seed for reproducible randomness.

    Returns
    -------
    LinearGaussianSystemSyms
        System with A, H, Sigma_p (Q), Sigma_m (R), and initial state x0.
    """
    A = np.array([[1.0, dt],
                  [0.0, 1.0]], dtype=float)

    # Measure position only (scalar)
    H = np.array([[1.0, 0.0]], dtype=float)  # shape (1,2)

    # Discrete CV process noise covariance (from white-acceleration model)
    Q = q * np.array([[dt**3/3.0, dt**2/2.0],
                      [dt**2/2.0, dt      ]], dtype=float)

    # Measurement noise covariance (scalar)
    R = np.array([[r**2]], dtype=float)

    x0 = np.asarray(x0, dtype=float)
    if x0.shape != (2,):
        raise ValueError(f"`x0` must be shape (2,), got {x0.shape}.")

    rng = np.random.default_rng(seed)
    return sims.LinearGaussianSystemSyms(A=A, H=H, Sigma_p=Q, Sigma_m=R, x0=x0, rng=rng)

# Must have p=1 in your system (scalar measurement). n can be >1.
sys = make_cv1d(dt=0.1, q=5e-2, r=1.0, x0=(0.0, 0.5), seed=7)  # p=1
# Run with simulated Y (T provided)
sys.animate_measurement_gaussians_scalar(T=80, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
                                         frame_ms=120, save_html_path=None, show=True)

# Or if you've already collected Y (shape (T,) or (T,1)):
# _, Y = sys.simulate(T=100)
# sys.animate_measurement_gaussians_scalar(Y=Y, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
#                                          save_html_path="kf_scalar_y_gaussians.html", auto_play=False)

## Simulation 2D-example

We consider a two-dimensional constant-velocity dynamical system. The hidden state at time step $k$ is

$$
x_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)\\
v_x(k)\\
v_y(k)
\end{bmatrix},
$$

where $(p_x(k),p_y(k))$ denote the position components and $(v_x(k),v_y(k))$ denote the velocity components.

The measurement consists only of the two position components:

$$
y_k =
\begin{bmatrix}
p_x^{\mathrm{meas}}(k)\\
p_y^{\mathrm{meas}}(k)
\end{bmatrix}.
$$

The linear Gaussian state-space model is

$$
x_k = A x_{k-1} + G w_{k-1},
$$

$$
y_k = Hx_k + z_k,
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,\Sigma_p),
\qquad
z_k \sim \mathscr{N}(0,\Sigma_m).
$$

The process noise sequence $w_k$, measurement noise sequence $z_k$, and the initial state are assumed mutually independent.

---


Assuming a sampling interval $\Delta t$, the constant-velocity kinematic equations are

$$
p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1),
$$

$$
p_y(k) = p_y(k-1) + \Delta t\,v_y(k-1),
$$

$$
v_x(k) = v_x(k-1),
$$

$$
v_y(k) = v_y(k-1).
$$

Therefore, in matrix form,

$$
x_k =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}
x_{k-1}
+
G w_{k-1}.
$$

Thus,

$$
A =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

---

Since the measurement contains only the position components, we have

$$
y_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)
\end{bmatrix}
+
z_k.
$$

Equivalently,

$$
y_k = Hx_k + z_k,
$$

where

$$
H =
\begin{bmatrix}
1 & 0 & 0 & 0\\
0 & 1 & 0 & 0
\end{bmatrix}.
$$

The measurement noise is modeled as

$$
z_k \sim \mathscr{N}(0,\Sigma_m),
$$

with

$$
\Sigma_m = r^2 I_2.
$$

Here, $r$ is the standard deviation of the measurement noise in each position coordinate.

---


A common way to model uncertainty in a constant-velocity system is to assume that the unmodeled acceleration is random. Let

$$
w_{k-1} =
\begin{bmatrix}
a_x(k-1)\\
a_y(k-1)
\end{bmatrix},
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,q^2I_2).
$$

Here, $q$ controls the acceleration-noise intensity.

Over one sampling interval $\Delta t$, the random acceleration affects both position and velocity:

$$
p_x(k)
=
p_x(k-1)
+
\Delta t\,v_x(k-1)
+
\frac{1}{2}\Delta t^2 a_x(k-1),
$$

$$
p_y(k)
=
p_y(k-1)
+
\Delta t\,v_y(k-1)
+
\frac{1}{2}\Delta t^2 a_y(k-1),
$$

$$
v_x(k)
=
v_x(k-1)
+
\Delta t\,a_x(k-1),
$$

$$
v_y(k)
=
v_y(k-1)
+
\Delta t\,a_y(k-1).
$$

Therefore,

$$
G =
\begin{bmatrix}
\frac{1}{2}\Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^2\\
\Delta t & 0\\
0 & \Delta t
\end{bmatrix},
$$

and

$$
\Sigma_p = q I_2.
$$

The induced state-space process covariance is

$$
Q
=
G\Sigma_pG^T.
$$

Since $\Sigma_p=qI_2$, this becomes

$$
Q
=
qGG^T.
$$

Explicitly,

$$
Q
=
q
\begin{bmatrix}
\frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3 & 0\\
0 & \frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3\\
\frac{1}{2}\Delta t^3 & 0 & \Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^3 & 0 & \Delta t^2
\end{bmatrix}.
$$

---

Thus, the full Gaussian filter is

$$
x^-_k =
Ax^+_{k-1}+Gw_{k-1},
\qquad
w_{k-1}\sim\mathscr{N}(0,q^2I_2),
$$

$$
y^-_k =
Hx^-_k+z_k,
\qquad
z_k\sim\mathscr{N}(0,r^2I_2).
$$

In [ ]:
# 2D position-only measurements (p=2), CV model in x & y
def make_cv2d(
    dt=0.1,
    q=1e-4,
    r=0.5,
    x0=(0, 0, 1, 0.5),
    seed=0,
    *,
    use_G=True,
    noise_model="accel_white",
):
    """
    Build a 2D constant-velocity (CV) linear-Gaussian system.

    States: x = [x, y, vx, vy]^T
      A = [[1, 0, dt, 0 ],
           [0, 1, 0 , dt],
           [0, 0, 1 , 0 ],
           [0, 0, 0 , 1 ]]

    Measurements: y = [x, y]^T  (position only)

    Process-noise options
    ---------------------
    - use_G=True, noise_model='accel_white'  (default):
        Uses an explicit G that maps a 2D white acceleration noise (w ~ N(0, q I_2))
        into the state:
            G = [[dt^2/2,     0   ],
                 [   0  ,  dt^2/2],
                 [  dt ,     0   ],
                 [   0 ,    dt   ]]
        Sigma_p = q * I_2   (in w-space)
        → State-space covariance is G Sigma_p G^T
        (This is the common “white-acceleration” CV model.)

    - use_G=False:
        Legacy behavior (no G). We provide the classic state-space Q directly
        corresponding to (velocity random-walk discretization):
            Q_block = [[dt^3/3, dt^2/2],
                       [dt^2/2, dt     ]] * q
        Q = blockdiag(Q_block, Q_block)
        Sigma_p = Q  (already in state space), G=None

    Notes
    -----
    The two variants imply slightly different discrete-time process covariances.
    Pick the one that matches your physical assumption / reference text.
    """
    A = np.array([
        [1, 0, dt,  0],
        [0, 1,  0, dt],
        [0, 0,  1,  0],
        [0, 0,  0,  1],
    ])
    H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ])
    R = np.eye(2) * (r**2)

    if use_G:
        if noise_model != "accel_white":
            raise ValueError("When use_G=True, supported noise_model is only 'accel_white'.")
        # White acceleration injected into vx, vy
        G = np.array([
            [0.5*dt*dt, 0.0       ],
            [0.0      , 0.5*dt*dt],
            [dt       , 0.0       ],
            [0.0      , dt        ],
        ])
        Sigma_p = np.eye(2) * (q**2)  # w-space covariance (2x2)
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Sigma_p, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )
    else:
        # Legacy/state-space Q (velocity random-walk discretization)
        Q_block = np.array([[dt**3/3, dt**2/2],
                            [dt**2/2, dt      ]], dtype=float) * q
        Q = np.block([
            [Q_block,               np.zeros((2, 2))],
            [np.zeros((2, 2)),      Q_block        ],
        ])
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Q, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )

In [ ]:
sys = make_cv2d(dt=0.1, q=1e-2, r=0.5, x0=(0,0, 0.5, -0.2), seed=7)
X2, Y2 = sys.simulate(T=300)
sys.plot_y(Y2, nbins=40, component_labels=["pos_x", "pos_y"])

In [ ]:
# Option A: simulate 300 steps internally, starting from broad prior
M, Yhat= sys.filter_with_kf_and_plot(T=300, Y=Y2, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
                                      component_labels=["pos_x","pos_y"], show=True)

# Option B: if you already have measurements Y, just pass them:
# X, Y = sys.simulate(T=300)
# M, Yhat = sys.filter_with_kf_and_plot(Y=Y, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
#                                       component_labels=["pos_x","pos_y"])

## HVAC Example

### HVAC-Model

#### Zone Temperature (Sensible Energy)

\begin{align}
C_s \dot T_z =
- UA\,T_z - c_{pa}(m_{inf}+m_{sa})T_z
\ +\ UA\,T_o + c_{pa} m_{inf} T_o + c_{pa} m_{sa} T_{sa}
+ Q_{bg} + f_c\,q^{occ}_{sens} N
\end{align}
where:
- $C_s$ = effective sensible thermal capacitance $[J/K]$
- $UA$ = overall heat transfer conductance $[W/K]$
- $c_{pa}$ = specific heat of air $[\approx 1006~J/(kg\,K)]$
- $m_{inf}$, $m_{sa}$ = outdoor- infiltration/supply air flow rates $[kg/s]$
- $T_o$, $T_z$, $T_{sa}$ = outdoor, zone, supply air temperatures $[^\circ C]$
- $Q_{bg}$, $q^{occ}_{sens}$ = background and per-person sensible heat gains $[W]$
- $f_c$ = convective fraction of sensible internal gain
- $N$ = number of occupants


Define
\begin{align}
\alpha_{o}&=\frac{UA + c_{pa} m_{inf}}{C_s}\\
\alpha_s&=\frac{c_{pa}}{C_s}\\
\alpha_e&= \frac{Q_{bg} + f_c\,q^{occ}_{sens} N}{C_s}
\end{align}

Then we have
\begin{align}
\dot T_z =
- (\alpha_o+m_{sa}\alpha_s)T_z+\ \alpha_o T_o + m_{sa}\alpha_s T_{sa}
+ \alpha_e
\end{align}

#### Zone Humidity Ratio (Moisture)

\begin{align}
M\dot\omega_z =
- (m_{inf} + m_{sa}) \omega_z
\ +\ m_{inf} \omega_o + m_{sa} \omega_{sa}
 + G_{bg} + g^{occ}_{\omega} N
\end{align}
where:
- $M$ = zone dry air mass or effective moisture capacity $$[kg_{dry}]$$
- $\omega_o,\, \omega_z,\, \omega_{sa}$ = outdoor, zone, supply air humidity ratios $[kg/kg_{dry}]$
- $G_{bg}$, $g^{occ}_{\omega}$ = background and per-person vapor gains $$[kg/s]$$


Define
\begin{align}
\beta_{o}&=\frac{m_{inf}}{M}\\
\beta_s&=\frac{1}{M}\\
\beta_e&= \frac{G_{bg} + g^{occ}_{\omega} N}{M}
\end{align}

Then we have

\begin{align}
\dot\omega_z =
- (\beta_o+m_{sa}\beta_s) \omega_z
\ +\ \beta_o \omega_o + m_{sa}\beta_{s} \omega_{sa}
 + \beta_e
\end{align}

#### Zone $CO_2$ Concentration

\begin{align}
M\dot c_z =
- (m_{inf} + m_{sa}) c_z
\ +\ m_{inf} c_o + m_{sa} c_{sa}
 + g_{CO2}^{occ} N
\end{align}
where:
- $c_o,\, c_z,\, c_{sa}$ = outdoor, zone, supply air $CO_2$ concentrations $[kg/kg_{dry}]$
- $g^{occ}_{CO2}$ = per-person $CO_2$ generation rate $[kg/s]$.

And we also have

\begin{align}
\dot{c}_z &=
- (\beta_o+m_{sa}\beta_s) c_z
\ +\ \beta_o c_o + m_{sa}\beta_{s} c_{sa}
 + \gamma_e
\end{align}

where
\begin{align}
\gamma_e&= \frac{g_{CO2}^{occ} N}{M}
\end{align}



#### Dynamic Equations


\begin{align}
\dot T_z &=
- (\alpha_o+m_{sa}\alpha_s)T_z+\ \alpha_o T_o + m_{sa}\alpha_s T_{sa}
+ \alpha_e\\
\dot\omega_z &=
- (\beta_o+m_{sa}\beta_s) \omega_z
\ +\ \beta_o \omega_o + m_{sa}\beta_{s} \omega_{sa}
 + \beta_e\\
\dot{c}_z &=
- (\beta_o+m_{sa}\beta_s) c_z
\ +\ \beta_o c_o + m_{sa}\beta_{s} c_{sa}
 + \gamma_e
\end{align}

***

These forms are **physically complete, transparent, and directly express the conservation of energy, mass (moisture), and tracer (CO₂) for an air-conditioned zone** with standard HVAC inputs.



### Model Paramater Estimation

#### Kalman Filter Model


We will assume that $\{T_o, \omega_o, c_o, T_{sa},\omega_{sa},c_{sa}\}$ are measured accurately.

We will consider the augmented state
\begin{align}
x_k &\triangleq
\begin{bmatrix}\alpha_{o,k} & \alpha_{s,k} & \alpha_{e,k} & \beta_{o,k} & \beta_{s,k} & \beta_{e,k} & \gamma_{e,k}
& T_{z,k} & \omega_{z,k} & c_{z,k}
\end{bmatrix}^T
\end{align}

The discrete time evolution of the system is then:
\begin{align}
x_k & = f(x_{k-1})+w_k, \qquad w_k \sim \mathscr{N}(0,\Sigma_Q),\\
y_k & = H_k x_k + \varepsilon_k, \qquad \varepsilon_k \sim \mathscr{N}(0,\Sigma_R)
\end{align}
where
\begin{align}
f({x}_{k-1}) = {x}_{k-1}+\Delta t\begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ -({\alpha}_{o,k-1}+m_{sa,k-1}{\alpha}_{s,k-1}){T}_{z,k-1} + {\alpha}_{o,k-1}T_{o,k-1} + m_{sa,k-1}{\alpha}_{s,k-1}T_{sa,k-1} + {\alpha}_{e,k-1}\\
-({\beta}_{o,k-1}+m_{sa,k-1}{\beta}_{s,k-1}){\omega}_{z,k-1} + {\beta}_{o,k-1}\omega_{o,k-1} + m_{sa,k-1}{\beta}_{s,k-1}\omega_{sa,k-1} + {\beta}_{e,k-1}\\
-({\beta}_{o,k-1}+m_{sa,k-1}{\beta}_{s,k-1}){c}_{z,k-1} + {\beta}_{o,k-1|k-1}c_{o,k-1} + m_{sa,k-1}{\beta}_{s,k-1}c_{sa,k-1} + {\gamma}_{e,k-1} \end{bmatrix}\\
H_k & =
\begin{bmatrix}0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 & 0\\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0\\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1\\
\end{bmatrix}\\
y_k &= \begin{bmatrix}
T_{z,k}\\ \omega_{z,k} \\ c_{z,k}
\end{bmatrix}
\end{align}



#### The Extended Kalman Filter


1. Prediction Step
\begin{align}
\widehat{x}_{k|k-1}=f(\widehat{x}_{k-1|k-1})
\end{align}


2. Predict the error covariance

\begin{align}
P_{k|k-1}=F_{k-1}P_{k-1|k-1}F_{k-1}^{T}+\Sigma _{Q}
\end{align}

3. Compute the Kalman gain

\begin{align}
K_{k}=P_{k|k-1}H_{k}^{T}(H_{k}P_{k|k-1}H_{k}^{T}+\Sigma _{R})^{-1}
\end{align}

4. Update the state estimate
\begin{align}
\widehat{x}_{k|k}=\widehat{x}_{k|k-1}+K_{k}(y_{k}-H_{k}\widehat{x}_{k|k-1})
\end{align}

5. Update the error covariance
\begin{align}
P_{k|k}=(I-K_{k}H_{k})P_{k|k-1}
\end{align}


Here
\begin{align}
f(\widehat{x}_{k-1|k-1}) \triangleq \widehat{x}_{k-1|k-1}+\Delta t\begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ -(\widehat{\alpha}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\alpha}_{s,k-1|k-1})\widehat{T}_{z,k-1|k-1} + \widehat{\alpha}_{o,k-1|k-1}T_{o,k-1} + m_{sa,k-1}\widehat{\alpha}_{s,k-1|k-1}T_{sa,k-1} + \widehat{\alpha}_{e,k-1|k-1}\\
-(\widehat{\beta}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1})\widehat{\omega}_{z,k-1|k-1} + \widehat{\beta}_{o,k-1|k-1}\omega_{o,k-1} + m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1}\omega_{sa,k-1} + \widehat{\beta}_{e,k-1|k-1}\\
-(\widehat{\beta}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1})\widehat{c}_{z,k-1|k-1} + \widehat{\beta}_{o,k-1|k-1}c_{o,k-1} + m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1}c_{sa,k-1} + \widehat{\gamma}_{e,k-1|k-1} \end{bmatrix}
\end{align}
and the Jacobian at $\widehat{x}_{k-1|k-1}$ is
\begin{align}
F_{k-1} &= I+ \Delta t\begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
-\widehat{T}_{z,k-1|k-1}+T_{o,k-1} & -m_{sa,k-1}\widehat{T}_{z,k-1|k-1}+m_{sa,k-1}T_{sa,k-1} & 1 & 0 & 0 & 0 & 0 & -(\widehat{\alpha}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\alpha}_{s,k-1|k-1}) & 0 & 0 \\
0 & 0 & 0 & -\widehat{\omega}_{z,k-1|k-1}+\omega_{o,k-1} & -m_{sa,k-1}\widehat{\omega}_{z,k-1|k-1}+m_{sa,k-1}\omega_{sa,k-1} & 1 & 0 & 0 & -(\widehat{\beta}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1}) & 0 \\
0 & 0 & 0 & -\widehat{c}_{z,k-1|k-1}+c_{o,k-1} & -m_{sa,k-1}\widehat{c}_{z,k-1|k-1}+m_{sa,k-1}c_{sa,k-1} & 0 & 1 & 0 & 0 & -(\widehat{\beta}_{o,k-1|k-1}+m_{sa,k-1}\widehat{\beta}_{s,k-1|k-1}) \end{bmatrix}
\end{align}

#### Kalman Filter Implementation

In [ ]:
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def run_hvac_ekf_and_plot(
    Y_meas,          # (T, 3) array of measured [T_z, omega_z, c_z]
    U_inputs,        # (T, 5) array of inputs [m_sa, T_o, T_sa, omega_o, omega_sa, c_o, c_sa]
    x0_init,         # (10,) initial augmented state guess
    P0_init,         # (10, 10) initial covariance
    Sigma_Q,         # (10, 10) process noise covariance matrix
    Sigma_R,         # (3, 3) measurement noise covariance matrix
    dt,              # time step in seconds (e.g., 60.0)
    kf=None          # instance of LinearKF
):
    """
    Runs an Extended Kalman Filter (EKF) for the 10D HVAC state-space model:
      x = [alpha_o, alpha_s, alpha_e, beta_o, beta_s, beta_e, gamma_e, T_z, omega_z, c_z]^T
    """
    if kf is None:
        kf = sims.LinearKF(use_joseph=True, symmetrize=True)

    T = Y_meas.shape[0]
    n = 10  # State dimension
    p = 3   # Measurement dimension (T_z, omega_z, c_z)

    # Measurement matrix H is static (selects the last 3 state variables)
    H = np.zeros((p, n))
    H[0, 7] = 1.0  # T_z
    H[1, 8] = 1.0  # omega_z
    H[2, 9] = 1.0  # c_z

    # Storage arrays
    X_est = np.zeros((T, n))
    Y_est = np.zeros((T, p))
    P_trace = np.zeros(T)
    P_diag = np.zeros((T, n))

    x_curr = np.asarray(x0_init, dtype=float).copy()
    P_curr = np.asarray(P0_init, dtype=float).copy()

    for k in range(T):
        # Extract inputs for step k-1
        m_sa, T_o, T_sa, w_o, w_sa, c_o, c_sa = U_inputs[k]

        # --- 1. Compute Continuous Dynamics & Jacobian (F_k-1) ---
        a_o, a_s, a_e, b_o, b_s, b_e, g_e, T_z, w_z, c_z = x_curr

        # Dynamic rates f_cont(x)
        dT_z = -(a_o + m_sa * a_s) * T_z + a_o * T_o + m_sa * a_s * T_sa + a_e
        dw_z = -(b_o + m_sa * b_s) * w_z + b_o * w_o + m_sa * b_s * w_sa + b_e
        dc_z = -(b_o + m_sa * b_s) * c_z + b_o * c_o + m_sa * b_s * c_sa + g_e

        # Discrete prediction: x_pred = x + dt * f_cont(x)
        x_pred = x_curr.copy()
        x_pred[7] += dt * dT_z
        x_pred[8] += dt * dw_z
        x_pred[9] += dt * dc_z

        # Continuous Jacobian Matrix J = df/dx
        J = np.zeros((n, n))
        # dT_z derivatives
        J[7, 0] = T_o - T_z              # d/d(alpha_o)
        J[7, 1] = m_sa * (T_sa - T_z)    # d/d(alpha_s)
        J[7, 2] = 1.0                    # d/d(alpha_e)
        J[7, 7] = -(a_o + m_sa * a_s)    # d/d(T_z)

        # dw_z derivatives
        J[8, 3] = w_o - w_z              # d/d(beta_o)
        J[8, 4] = m_sa * (w_sa - w_z)    # d/d(beta_s)
        J[8, 5] = 1.0                    # d/d(beta_e)
        J[8, 8] = -(b_o + m_sa * b_s)    # d/d(omega_z)

        # dc_z derivatives
        J[9, 3] = c_o - c_z              # d/d(beta_o)
        J[9, 4] = m_sa * (c_sa - c_z)    # d/d(beta_s)
        J[9, 6] = 1.0                    # d/d(gamma_e)
        J[9, 9] = -(b_o + m_sa * b_s)    # d/d(c_z)

        # Linearized State Transition Matrix: A_k = I + dt * J
        A_k = np.eye(n) + dt * J

        # --- 2. Step EKF using LinearKF ---
        # Note: We directly pass x_pred into measurement update or run custom step
        m_pred, P_pred = kf.predict(x_curr, P_curr, A_k, Sigma_Q)
        # Correct the predicted state mean with non-linear mapping
        m_pred = x_pred

        m_upd, P_upd, K, S = kf.measurement_update(m_pred, P_pred, H, Y_meas[k], Sigma_R)

        # Update state and covariance
        x_curr = m_upd
        P_curr = P_upd

        # Store estimates
        X_est[k] = x_curr
        Y_est[k] = H @ x_curr
        P_trace[k] = np.trace(P_curr)
        P_diag[k] = np.diag(P_curr)

    t = np.arange(T) * dt / 3600.0  # Time in hours

    # =========================================================================
    # Plot 1: Estimated y_k vs Measured y_k
    # =========================================================================
    fig_y = make_subplots(rows=3, cols=1, shared_xaxes=True,
                          subplot_titles=["Zone Temperature (T_z)",
                                          "Zone Humidity Ratio (ω_z)",
                                          "Zone CO2 Concentration (c_z)"])

    meas_labels = ["T_z [°C]", "ω_z [kg/kg]", "c_z [ppm]"]
    for i in range(p):
        fig_y.add_trace(go.Scatter(x=t, y=Y_meas[:, i], mode="markers",
                                   name=f"Measured {meas_labels[i]}", marker=dict(size=4)), row=i+1, col=1)
        fig_y.add_trace(go.Scatter(x=t, y=Y_est[:, i], mode="lines",
                                   name=f"Estimated {meas_labels[i]}", line=dict(width=2)), row=i+1, col=1)
        fig_y.update_yaxes(title_text=meas_labels[i], row=i+1, col=1)

    fig_y.update_xaxes(title_text="Time [Hours]", row=3, col=1)
    fig_y.update_layout(title="Measured vs. EKF Estimated Outputs (y_k)", template="plotly_white", height=700)
    fig_y.show()

    # =========================================================================
    # Plot 2: Estimated x_k Parameters (without y_k parts)
    # =========================================================================
    param_names = [r"$\alpha_o$", r"$\alpha_s$", r"$\alpha_e$",
                   r"$\beta_o$", r"$\beta_s$", r"$\beta_e$", r"$\gamma_e$"]

    fig_x = make_subplots(rows=7, cols=1, shared_xaxes=True,
                          subplot_titles=[f"Parameter {p_name}" for p_name in param_names])

    for j in range(7):
        fig_x.add_trace(go.Scatter(x=t, y=X_est[:, j], mode="lines",
                                   name=param_names[j], line=dict(color="crimson")), row=j+1, col=1)
        fig_x.update_yaxes(title_text=param_names[j], row=j+1, col=1)

    fig_x.update_xaxes(title_text="Time [Hours]", row=7, col=1)
    fig_x.update_layout(title="EKF Online Estimated Zone Parameters (Structural/Load)", template="plotly_white", height=1200)
    fig_x.show()

    # =========================================================================
    # Plot 3: Measures of Uncertainty / Covariance (P_k)
    # =========================================================================
    fig_cov = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=["Overall Uncertainty Trace: tr(P_k)",
                                            "Parameter Covariance Variances (Diag of P_k)"])

    # Trace of P
    fig_cov.add_trace(go.Scatter(x=t, y=P_trace, mode="lines", name="tr(P_k)",
                                 line=dict(color="black", width=2)), row=1, col=1)

    # Individual parameter variances (Log scale for clarity)
    for j in range(7):
        fig_cov.add_trace(go.Scatter(x=t, y=P_diag[:, j], mode="lines",
                                     name=f"Var({param_names[j]})"), row=2, col=1)

    fig_cov.update_yaxes(title_text="tr(P)", type="log", row=1, col=1)
    fig_cov.update_yaxes(title_text="Variance", type="log", row=2, col=1)
    fig_cov.update_xaxes(title_text="Time [Hours]", row=2, col=1)
    fig_cov.update_layout(title="Covariance & Uncertainty Metrics over Time", template="plotly_white", height=600)
    fig_cov.show()

    return X_est, Y_est, P_diag

##### Simulation with Real Data

In [ ]:
!pip install "git+https://github.com/mugalan/data-analysis-tool.git"

In [ ]:
from data_analysis import DataInspector, PlottingMethods
inspector = DataInspector()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pooriamst/occupancy-detection")

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df=pd.read_csv(path+"/datatest.csv")


In [ ]:
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Load Data & Convert Relative Humidity to Humidity Ratio (omega_z)
# -----------------------------------------------------------------------------
df = pd.read_csv(path + "/datatest.csv")

# If HumidityRatio isn't already present, we compute it via Tetens equation
if "HumidityRatio" in df.columns:
    df['omega_z'] = df['HumidityRatio']
else:
    # Tetens equation for saturation vapor pressure
    P_sat = 0.61078 * np.exp((17.27 * df['Temperature']) / (df['Temperature'] + 237.3))
    P_v = (df['Humidity'] / 100.0) * P_sat
    df['omega_z'] = 0.622 * P_v / (101.325 - P_v)

# -----------------------------------------------------------------------------
# 2. Extract Measurement Matrix Y_meas (T_z, omega_z, c_z)
# -----------------------------------------------------------------------------
Y_meas = df[['Temperature', 'omega_z', 'CO2']].values
T = Y_meas.shape[0]
dt = 60.0  # 1-minute sample intervals in seconds

# -----------------------------------------------------------------------------
# 3. Construct Input Vector Matrix U_inputs (7 inputs per time step)
#    [m_sa, T_o, T_sa, omega_o, omega_sa, c_o, c_sa]
# -----------------------------------------------------------------------------
# Standard office operational defaults (Adjust based on exact setup if known):
U_inputs = np.zeros((T, 7))

# Estimate ventilation airflow (higher flow during active/occupied hours)
if "Occupancy" in df.columns:
    U_inputs[:, 0] = np.where(df['Occupancy'] == 1, 0.35, 0.05)  # m_sa (kg/s)
else:
    U_inputs[:, 0] = 0.2  # Nominal default supply flow rate

U_inputs[:, 1] = 22.0     # Outdoor Temp T_o (°C)
U_inputs[:, 2] = 16.0     # Supply Air Temp T_sa (°C)
U_inputs[:, 3] = 0.008    # Outdoor Humidity Ratio omega_o (kg/kg)
U_inputs[:, 4] = 0.005    # Supply Humidity Ratio omega_sa (kg/kg)
U_inputs[:, 5] = 400.0    # Outdoor CO2 c_o (ppm)
U_inputs[:, 6] = 400.0    # Supply CO2 c_sa (ppm)

# -----------------------------------------------------------------------------
# 4. Set Initial Guesses and Covariances
# -----------------------------------------------------------------------------
# State Vector: [alpha_o, alpha_s, alpha_e, beta_o, beta_s, beta_e, gamma_e, T_z, omega_z, c_z]
x0_init = np.array([
    1.0e-5,        # alpha_o
    5.0e-4,        # alpha_s
    1.0e-3,        # alpha_e
    1.0e-5,        # beta_o
    5.0e-4,        # beta_s
    1.0e-6,        # beta_e
    5.0e-3,        # gamma_e
    Y_meas[0, 0],  # T_z (initial measured temp)
    Y_meas[0, 1],  # omega_z (initial measured humidity ratio)
    Y_meas[0, 2]   # c_z (initial measured CO2)
])

# Initial error covariance P0
P0_init = np.diag([1e-8, 1e-6, 1e-4, 1e-8, 1e-6, 1e-10, 1e-4, 1e-2, 1e-6, 1.0])

# Process noise covariance Sigma_Q (enables parameter tracking)
Sigma_Q = np.diag([1e-12, 1e-10, 1e-8, 1e-12, 1e-10, 1e-14, 1e-8, 1e-4, 1e-8, 1e-2])

# Measurement noise covariance Sigma_R (Temperature, Humidity Ratio, CO2)
Sigma_R = np.diag([0.1**2, (1e-4)**2, 5.0**2])

# -----------------------------------------------------------------------------
# 5. Call run_hvac_ekf_and_plot
# -----------------------------------------------------------------------------
kf_instance = sims.LinearKF(use_joseph=True, symmetrize=True)

X_est, Y_est, P_diag = run_hvac_ekf_and_plot(
    Y_meas=Y_meas,
    U_inputs=U_inputs,
    x0_init=x0_init,
    P0_init=P0_init,
    Sigma_Q=Sigma_Q,
    Sigma_R=Sigma_R,
    dt=dt,
    kf=kf_instance
)

## Rigid Body Motion Estimation

[Refer to this note for a comprehensive treatment of Kalman Filters for Rigid Body Motion estimation](https://github.com/mugalan/intrinsic-rigid-body-control-estimation/blob/main/intrinsic-DEKF/RigidBodyIntinsicEKF_DHSM.ipynb)